In [12]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

In [13]:
df = pd.read_csv(r"C:\Users\Aanjney\Desktop\Deploy\Student Productivity (with EDA)\student_productivity_distraction_dataset_20000.csv")
df.head()

,student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score
0,1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78
1,2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.90,48.99
2,3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.60
3,4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87
4,5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.90


In [14]:
df.drop(columns=["student_id"], inplace=True)

In [15]:
target = "productivity_score"

X = df.drop(columns=[target])
y = df[target]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
numeric_features = X.select_dtypes(include=['int64','float64']).columns
categorical_features = ['gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

In [18]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForest": RandomForestRegressor(),
    "GradientBoosting": GradientBoostingRegressor(),
    "SVR": SVR()
}

param_grid = {
    "Ridge": {"model__alpha": [0.1, 1, 10]},
    "Lasso": {"model__alpha": [0.01, 0.1, 1]},
    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20]
    },
    "GradientBoosting": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.01, 0.1]
    },
    "SVR": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ['rbf']
    }
}

best_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])
best_score = -np.inf
results = []

In [19]:
for name, model in models.items():

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    if name in param_grid:
        grid = GridSearchCV(
            pipe,
            param_grid[name],
            cv=5,
            scoring='r2',
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        final_model = grid.best_estimator_
    else:
        final_model = pipe.fit(X_train, y_train)

    # Cross-validation
    cv_scores = cross_val_score(final_model, X_train, y_train, cv=5, scoring='r2')

    y_pred = final_model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append([name, r2, mae, rmse, cv_scores.mean()])

    if r2 > best_score:
        best_score = r2
        best_model = final_model


In [20]:
results_df = pd.DataFrame(results,
                          columns=["Model","R2","MAE","RMSE","CV_R2"])

print(results_df.sort_values(by="R2", ascending=False))


              Model        R2       MAE      RMSE     CV_R2
0  LinearRegression  1.000000  0.002483  0.002873  1.000000
1             Ridge  1.000000  0.002486  0.002876  1.000000
2             Lasso  0.999998  0.019989  0.024735  0.999998
5               SVR  0.998945  0.296894  0.521427  0.998712
4  GradientBoosting  0.996318  0.770999  0.973907  0.996117
3      RandomForest  0.975377  1.980038  2.518487  0.973360


In [21]:
joblib.dump(best_model, "best_model.pkl")
print("Best model saved as best_model.pkl")

Best model saved as best_model.pkl
